## Cell 1 — Environment Bootstrap

In [ ]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Cell 2 — Configuration

In [ ]:
import os, gc, json
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'artifacts').exists():
            return candidate
    raise RuntimeError('Could not locate project root with configs/ and artifacts/.')


import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Environment detection ─────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # Upload all_speakers.csv as a Kaggle dataset input named "speeches"
    DATA_CSV = Path('//kaggle/input/datasets/himathdhanapala/speeches/all_speakers.csv')
    OUT_DIR  = Path('/kaggle/working/embeddings')
else:
    BASE_DIR = find_project_root()
    DATA_CSV = BASE_DIR / 'artifacts' / 'final_v14' / 'all_speakers.csv'
    OUT_DIR  = BASE_DIR / 'embeddings'

OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment : {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'Device      : {DEVICE}')
print(f'DATA_CSV    : {DATA_CSV}')
print(f'OUT_DIR     : {OUT_DIR}')

MODEL_NAME  = 'BAAI/bge-m3'
MAX_SEQ_LEN = 8192          # BGE-M3 native limit
BATCH_SIZE  = 1
YEARS       = list(range(2019, 2027))   # 2019 -> 2026  (change as needed)
RESUME      = True           # skip year if .npy already exists

## Cell 3 — Load & Preview CSV

In [ ]:
df = pd.read_csv(DATA_CSV)
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year

print(f'Total speeches : {len(df)}')
print('\nYear distribution:')
print(df[df['year'].isin(YEARS)]['year'].value_counts().sort_index().to_string())

## Cell 4 — Load Model

In [ ]:
print(f'Loading {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = MAX_SEQ_LEN
print(f'Model loaded  max_seq_length={model.max_seq_length}')

## Cell 5 — Encode Year by Year

In [ ]:
year_shapes = {}

for yr in YEARS:
    out_path = OUT_DIR / f'{yr}.npy'

    if RESUME and out_path.exists():
        arr = np.load(out_path)
        year_shapes[yr] = arr.shape
        print(f'[SKIP] {yr}: already exists {arr.shape}')
        continue

    df_yr = df[df['year'] == yr].reset_index(drop=True)
    if len(df_yr) == 0:
        print(f'[SKIP] {yr}: no speeches in CSV')
        continue

    texts = df_yr['text'].fillna('').tolist()
    print(f'\n[ENCODE] {yr}: {len(texts)} speeches ...')

    emb = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(out_path, emb)
    year_shapes[yr] = emb.shape
    print(f'  Saved {out_path.name}  shape={emb.shape}')

    # Free GPU memory between years
    del emb
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print('\nDone encoding all years')
print(year_shapes)

## Cell 6 — Build Master Embeddings File

In [ ]:
master_path = OUT_DIR / 'master_embeddings_2017_2026.npy'

year_files = sorted(
    f for f in OUT_DIR.glob('20*.npy')
    if 'master' not in f.name
)

print('Stacking per-year files:')
parts = []
for f in year_files:
    arr = np.load(f)
    parts.append(arr)
    print(f'  {f.name}: {arr.shape}')

master = np.vstack(parts)
np.save(master_path, master)
print(f'\nMaster saved: {master_path.name}  shape={master.shape}')

## Cell 7 — Verification

In [ ]:
# Verify each year file aligns with its CSV rows
print('Alignment check:')
ok, skip = [], []

for f in sorted(f for f in OUT_DIR.glob('20*.npy') if 'master' not in f.name):
    yr    = int(f.stem)
    emb   = np.load(f)
    csv_n = len(df[df['year'] == yr])
    status = 'OK' if emb.shape[0] == csv_n else 'MISMATCH'
    sym    = 'v' if status == 'OK' else 'x'
    print(f'  [{sym}] {yr}: emb={emb.shape[0]}  csv={csv_n}  [{status}]')
    (ok if status == 'OK' else skip).append(yr)

print(f'\nMatched : {ok}')
print(f'Skipped : {skip}')